In [3]:
# 기본
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 경고 뜨지 않게 설정
import warnings
warnings.filterwarnings('ignore')

# 그래프 설정
sns.set()

# 그래프 기본 설정
plt.rcParams['font.family'] = 'Malgun Gothic'
# plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['figure.figsize'] = 12, 6
plt.rcParams['font.size'] = 14
plt.rcParams['axes.unicode_minus'] = False

# 데이터 전처리 알고리즘
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

# 학습용과 검증용으로 나누는 함수
from sklearn.model_selection import train_test_split

# 교차 검증
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import cross_validate
from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold

# 평가함수
# 분류용
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import roc_auc_score

# 회귀용
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error

# 모델의 최적의 하이퍼 파라미터를 찾기 위한 도구
from sklearn.model_selection import GridSearchCV

# 머신러닝 알고리즘 - 분류
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import VotingClassifier

# 머신러닝 알고리즘 - 회귀
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.linear_model import ElasticNet
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import GradientBoostingRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.ensemble import VotingRegressor

# 차원 축소
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

# 군집
from sklearn.cluster import KMeans
from sklearn.cluster import MeanShift
from sklearn.cluster import estimate_bandwidth

# 학습 모델 저장을 위한 라이브러리
import pickle

In [4]:
target1=pd.read_parquet(r'train/1.회원정보/201807_train_.parquet')
target2=pd.read_parquet(r'train/1.회원정보/201808_train_.parquet')
target3=pd.read_parquet(r'train/1.회원정보/201809_train_.parquet')
target4=pd.read_parquet(r'train/1.회원정보/201810_train_.parquet')
target5=pd.read_parquet(r'train/1.회원정보/201811_train_.parquet')
target6=pd.read_parquet(r'train/1.회원정보/201812_train_.parquet')

In [5]:
all_df = pd.concat([target1, target2, target3, target4, target5, target6])
all_df.reset_index(inplace=True, drop=True)
all_df

,기준년월,ID,남녀구분코드,연령,Segment,회원여부_이용가능,회원여부_이용가능_CA,회원여부_이용가능_카드론,소지여부_신용,소지카드수_유효_신용,...,할인금액_제휴연회비_B0M,청구금액_기본연회비_B0M,청구금액_제휴연회비_B0M,상품관련면제카드수_B0M,임직원면제카드수_B0M,우수회원면제카드수_B0M,기타면제카드수_B0M,카드신청건수,Life_Stage,최종카드발급경과월
0,201807,TRAIN_000000,2,40대,D,1,1,0,1,1,...,0,0,0,0개,0개,0개,0개,0,자녀성장(2),22
1,201807,TRAIN_000001,1,30대,E,1,1,1,1,1,...,0,0,0,0개,0개,0개,0개,0,자녀성장(1),18
2,201807,TRAIN_000002,1,30대,C,1,1,0,1,1,...,0,0,0,0개,0개,0개,0개,0,자녀출산기,20
3,201807,TRAIN_000003,2,40대,D,1,1,0,1,2,...,0,0,0,0개,0개,0개,0개,1,자녀성장(2),17
4,201807,TRAIN_000004,2,40대,E,1,1,1,1,1,...,0,0,0,0개,0개,0개,0개,1,자녀성장(1),15
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2399995,201812,TRAIN_399995,2,70대이상,E,1,1,1,1,1,...,0,0,0,0개,0개,0개,0개,0,노년생활,39
2399996,201812,TRAIN_399996,2,50대,D,1,1,1,1,1,...,0,0,0,0개,0개,0개,0개,0,자녀성장(2),24
2399997,201812,TRAIN_399997,1,30대,C,1,1,0,1,1,...,0,0,0,0개,0개,0개,0개,0,자녀출산기,18
2399998,201812,TRAIN_399998,1,40대,E,1,1,1,1,1,...,0,0,0,0개,0개,0개,0개,0,자녀성장(1),27


In [9]:
all_df['최종카드발급일자']

0          20160912.0
1          20170122.0
2          20161113.0
3          20170205.0
4          20170409.0
              ...    
2399995    20150902.0
2399996    20161201.0
2399997    20170609.0
2399998    20160928.0
2399999    20170608.0
Name: 최종카드발급일자, Length: 2400000, dtype: float64

In [8]:
selected_df1 = all_df[[
    'ID',
    '기준년월',
    '소지카드수_이용가능_신용',
    '소지카드수_유효_신용',
    '이용가능여부_해외겸용_본인',
    '_2순위신용체크구분',
    '보유여부_해외겸용_본인',
    '수신거부여부_TM',
    '수신거부여부_메일',
    '수신거부여부_DM',
    '이용금액_R3M_신용체크',
    '이용금액_R3M_신용',
    '_1순위카드이용금액',
    '이용카드수_신용체크',
    '_2순위카드이용금액',
    '_1순위카드이용건수',
    '_2순위카드이용건수'
]].copy()

In [9]:
selected_df1.to_parquet('1.회원정보(train).parquet', index=False)

In [9]:
# 분석용 패키지 임포트
import pandas as pd
import numpy as np
from scipy.stats import f_oneway
import seaborn as sns
import matplotlib.pyplot as plt

# 범주형 상관계수용 함수 (Cramér's V)
from scipy.stats import chi2_contingency

def cramers_v(confusion_matrix):
    chi2 = chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum().sum()
    phi2 = chi2 / n
    r, k = confusion_matrix.shape
    phi2corr = max(0, phi2 - ((k-1)*(r-1))/(n-1))
    rcorr = r - ((r-1)**2)/(n-1)
    kcorr = k - ((k-1)**2)/(n-1)
    return np.sqrt(phi2corr / min((kcorr-1), (rcorr-1)))

# 수치형 중 범주형으로 처리할 컬럼 지정
manual_cat_cols = [
    '성별', '동의여부', '남녀구분코드', '연령', 'Life_Stage',
    '가입통신회사코드', '거주시도명', '직장시도명',
    '회원여부_이용가능', '회원여부_이용가능_CA', '회원여부_이용가능_카드론',
    '회원여부_연체', '탈퇴횟수_누적', '탈회회수_발급6개월이내', '탈회횟수_발급1년이내',
    '소지여부_신용', '소지카드수_유효_신용', '소지카드수_이용가능_신용',
    '보유여부_해외겸용_본인', '이용가능여부_해외겸용_본인', '이용여부_3M_해외겸용_보인',
    '연회비발생카드수_BOM', '_1순위신용체크구분', '_2순위신용체크구분',
    '이용거절여부_카드론', '동의여부_한도증액안내',
    '수신거부여부_TM', '수신거부여부_DM', '수신거부여부_메일', '수신거부여부_SMS',
    '마케팅동의여부'
]

# 수치형 컬럼
num_cols = all_df.select_dtypes(include=[np.number]).columns.tolist()
num_cols = [col for col in num_cols if col not in manual_cat_cols]

# 범주형 컬럼
cat_cols_auto = all_df.select_dtypes(exclude=[np.number]).columns.tolist()
cat_cols = cat_cols_auto + manual_cat_cols

# 중복 제거하면서 순서 유지
cat_cols = list(dict.fromkeys(cat_cols))

# 결과 리스트
anova = []
chi = []

# 컬럼 반복
for col in all_df.columns:
    if col == 'Segment':
        continue

    if col in num_cols:
        data = all_df[[col, 'Segment']].dropna()
        groups = [data[data['Segment'] == val][col] for val in data['Segment'].unique()]
        try:
            stat = f_oneway(*groups).statistic
            ss_between = sum([(g.mean() - data[col].mean())**2 * len(g) for g in groups])
            ss_total = sum((data[col] - data[col].mean())**2)
            eta2 = ss_between / ss_total
            anova.append({'변수': col, '유형': '수치형', '계수종류': 'Eta²', '상관계수': eta2})
        except:
            continue

    elif col in cat_cols:
        contingency = pd.crosstab(all_df[col], all_df['Segment'])
        if contingency.shape[0] > 1 and contingency.shape[1] > 1:
            try:
                v = cramers_v(contingency)
                chi.append({'변수': col, '유형': '범주형', '계수종류': "Cramér's V", '상관계수': v})
            except:
                continue

# 결과 데이터프레임
result_df1 = pd.DataFrame(anova)
result_df2 = pd.DataFrame(chi)

# 정렬
result_df1 = result_df1.sort_values(by='상관계수', ascending=False).reset_index(drop=True)
result_df2 = result_df2.sort_values(by='상관계수', ascending=False).reset_index(drop=True)

# 출력
display(result_df1)
display(result_df2)

,변수,유형,계수종류,상관계수
0,이용금액_R3M_신용체크,수치형,Eta²,0.38981
1,이용금액_R3M_신용,수치형,Eta²,0.34792
2,_1순위카드이용금액,수치형,Eta²,0.32973
3,이용카드수_신용체크,수치형,Eta²,0.16602
4,_2순위카드이용금액,수치형,Eta²,0.16594
5,_1순위카드이용건수,수치형,Eta²,0.15489
6,_2순위카드이용건수,수치형,Eta²,0.15100
7,이용카드수_신용,수치형,Eta²,0.14789
8,이용가능카드수_신용체크,수치형,Eta²,0.12901
9,이용가능카드수_신용,수치형,Eta²,0.12794


,변수,유형,계수종류,상관계수
0,ID,범주형,Cramér's V,0.91287
1,소지카드수_이용가능_신용,범주형,Cramér's V,0.20027
2,소지카드수_유효_신용,범주형,Cramér's V,0.17372
3,이용가능여부_해외겸용_본인,범주형,Cramér's V,0.16451
4,_2순위신용체크구분,범주형,Cramér's V,0.15418
5,보유여부_해외겸용_본인,범주형,Cramér's V,0.14787
6,수신거부여부_TM,범주형,Cramér's V,0.11045
7,수신거부여부_메일,범주형,Cramér's V,0.11013
8,수신거부여부_DM,범주형,Cramér's V,0.10848
9,수신거부여부_SMS,범주형,Cramér's V,0.09699


In [6]:
pd.set_option('display.max_rows', None)
pd.set_option('display.float_format', '{:.5f}'.format)
display(result_df1)
display(result_df2)

,변수,유형,계수종류,상관계수
0,이용금액_R3M_신용체크,수치형,Eta²,0.38981
1,이용금액_R3M_신용,수치형,Eta²,0.34792
2,_1순위카드이용금액,수치형,Eta²,0.32973
3,이용카드수_신용체크,수치형,Eta²,0.16602
4,_2순위카드이용금액,수치형,Eta²,0.16594
5,_1순위카드이용건수,수치형,Eta²,0.15489
6,_2순위카드이용건수,수치형,Eta²,0.15100
7,이용카드수_신용,수치형,Eta²,0.14789
8,이용가능카드수_신용체크,수치형,Eta²,0.12901
9,이용가능카드수_신용,수치형,Eta²,0.12794


,변수,유형,계수종류,상관계수
0,ID,범주형,Cramér's V,0.91287
1,소지카드수_이용가능_신용,범주형,Cramér's V,0.20027
2,소지카드수_유효_신용,범주형,Cramér's V,0.17372
3,이용가능여부_해외겸용_본인,범주형,Cramér's V,0.16451
4,_2순위신용체크구분,범주형,Cramér's V,0.15418
5,보유여부_해외겸용_본인,범주형,Cramér's V,0.14787
6,수신거부여부_TM,범주형,Cramér's V,0.11045
7,수신거부여부_메일,범주형,Cramér's V,0.11013
8,수신거부여부_DM,범주형,Cramér's V,0.10848
9,수신거부여부_SMS,범주형,Cramér's V,0.09699


In [7]:
with pd.ExcelWriter('5.잔액정보.xlsx', engine='openpyxl') as writer:
    result_df1.to_excel(writer, sheet_name='Result1', index=False)
    result_df2.to_excel(writer, sheet_name='Result2', index=False)


In [8]:
col = '평잔_일시불_6M'

if pd.api.types.is_numeric_dtype(all_df[col]):
    sns.boxplot(data=all_df, x='Segment', y=col)
else:
    sns.countplot(data=all_df, x=col, hue='Segment', order=all_df[col].value_counts().index)

plt.title(f"{col} vs Segment")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

KeyError: '평잔_일시불_6M'

In [ ]:
col = '월중평잔_일시불_B0M'

if pd.api.types.is_numeric_dtype(all_df[col]):
    sns.boxplot(data=all_df, x='Segment', y=col)
else:
    sns.countplot(data=all_df, x=col, hue='Segment', order=all_df[col].value_counts().index)

plt.title(f"{col} vs Segment")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
col = '잔액_일시불_B0M'

if pd.api.types.is_numeric_dtype(all_df[col]):
    sns.boxplot(data=all_df, x='Segment', y=col)
else:
    sns.countplot(data=all_df, x=col, hue='Segment', order=all_df[col].value_counts().index)

plt.title(f"{col} vs Segment")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# 대상 변수
col = '잔액_일시불_B0M'

# 이상치 비율 계산 함수
def get_outlier_ratio(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[column] < lower) | (df[column] > upper)]
    return len(outliers) / len(df)

# Segment별 이상치 비율 계산
outlier_ratios = (
    all_df.groupby('Segment')
    .apply(lambda group: get_outlier_ratio(group, col))
    .reset_index(name='이상치비율')
)

# 비율(%)로 변환
outlier_ratios['이상치비율'] *= 100

# 시각화
sns.barplot(data=outlier_ratios, x='Segment', y='이상치비율')
plt.title(f"Segment별 이상치 비율 - {col}")
plt.ylabel("이상치 비율 (%)")
plt.ylim(0, outlier_ratios['이상치비율'].max() * 1.1)
plt.tight_layout()
plt.show()

In [ ]:
col = '평잔_일시불_해외_6M'

if pd.api.types.is_numeric_dtype(all_df[col]):
    sns.boxplot(data=all_df, x='Segment', y=col, order=['A', 'B', 'C', 'D', 'E'])
else:
    sns.countplot(data=all_df, x=col, hue='Segment', order=all_df[col].value_counts().index)

plt.title(f"{col} vs Segment")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
# 대상 변수
col = '평잔_일시불_해외_6M'

# 이상치 비율 계산 함수
def get_outlier_ratio(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[column] < lower) | (df[column] > upper)]
    return len(outliers) / len(df)

# Segment별 이상치 비율 계산
outlier_ratios = (
    all_df.groupby('Segment')
    .apply(lambda group: get_outlier_ratio(group, col))
    .reset_index(name='이상치비율')
)

# 비율(%)로 변환
outlier_ratios['이상치비율'] *= 100

# 시각화
sns.barplot(data=outlier_ratios, x='Segment', y='이상치비율')
plt.title(f"Segment별 이상치 비율 - {col}")
plt.ylabel("이상치 비율 (%)")
plt.ylim(0, outlier_ratios['이상치비율'].max() * 1.1)
plt.tight_layout()
plt.show()